In [2]:
import pandas as pd 
import numpy as np
import random
import os 
import argparse
import json
import torch
import pickle
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from torch.utils.data import TensorDataset
from attrdict import AttrDict
from transformers import BertConfig, BertTokenizer, BertModel
from transformers import DistilBertModel, DistilBertTokenizer, DistilBertConfig, DistilBertForSequenceClassification
from transformers import RobertaModel, RobertaTokenizer, RobertaConfig, RobertaForSequenceClassification
from transformers import AdamW, get_linear_schedule_with_warmup

In [3]:
default_path = os.getcwd()
base_model = os.path.join('C:/Users/lamda/Desktop/LAMDA_git/EmoDep/base-model')
config_path = 'C:/Users/lamda/Desktop/LAMDA_git/EmoDep/config'
model_path = "C:/Users/lamda/Desktop/LAMDA_git/EmoDep/model/data_aug/seed"
config_file = "bert-base.json"
save_path = os.path.join(default_path, './')

In [213]:
criteria = 9; seed = 42

In [214]:
test_data = pd.read_csv(f'C:/Users/lamda/Desktop/LAMDA_git/EmoDep/data/bws/tagged/train-test/bws_a{criteria}_score_test.csv')

In [215]:
test_data

,text,translated,label
0,I have gone from literally being suicidal to l...,나는 말 그대로 자살에서 나 자신을 사랑하고 내 삶을 진정으로 사랑하는 것으로 바뀌었다,1
1,I was suicidal for a good chunk of last year a...,나는 작년의 상당 부분을 자살했고 나는 내가 그곳으로 다시 표류하는 것을 느낄 수 있다,9
2,At least when I kill myself you will get a nic...,"적어도 내가 자살할 때 당신은 좋은 보상을 받을 수 있을 거예요, 그렇죠?",4
3,I am uncertain if I will die without ever havi...,나는 내가 전에 키스를 한 번도 받지 않고 죽을지 아니면 처녀로 죽을지 확신할 수 없다,2
4,yelling at me saying i was a disappointment ...,내가 정말 자살하고 싶었다면 실망스럽다고 나에게 소리쳤다. 나는 더 열심히 노력했어...,6
...,...,...,...
315,For 4 years I had to sleep with a breathing ma...,4년 동안 나는 호흡기와 함께 자야 했다. 왜냐하면 내가 그렇게 하지 않으면 죽을 ...,3
316,Something similar happened to me recently and ...,"최근에 나에게 비슷한 일이 일어났고 나는 ""만약 내가 죽어야 한다는 신호가 있다면 ...",4
317,you may want to talk to your doc about the sui...,당신은 의사와 자살 생각에 대해 이야기하고 싶을 수도 있고 만약 당신이 과거에 자살...,2
318,I try to occupy myself by watching anime or re...,나는 애니메이션을 보거나 헛소리를 읽으며 나 자신을 차지하려고 노력하지만 내 생각과...,8


In [216]:
class BertDataset(Dataset):
    def __init__(self, data_file):
        self.data = data_file
    
    def __len__(self):
        return len(self.data.label)
    
    def reset_index(self):
        self.data.reset_index(inplace=True, drop=True)
    
    def __getitem__(self, idx):
        '''
        return text, label
        '''
        self.reset_index()
        text = self.data.text[idx]
        label = self.data.label[idx]
        return text, label

In [217]:
class BertProcessor():
    def __init__(self, config, training_config, tokenizer, truncation=True):
        self.tokenizer = tokenizer 
        self.max_len = config.max_position_embeddings
        self.pad = training_config.pad
        self.batch_size = training_config.train_batch_size
        self.truncation = truncation
    
    def convert_data(self, data_file):
        context2 = None    # single sentence classification
        batch_encoding = self.tokenizer.batch_encode_plus(
            [(data_file[idx][0], context2) for idx in range(len(data_file))],   # text, 
            max_length = self.max_len,
            padding = self.pad,
            truncation = self.truncation
        )
        
        features = []
        for i in range(len(data_file)):
            inputs = {k: batch_encoding[k][i] for k in batch_encoding}
            try:
                inputs['label'] = data_file[i][1] 
            except:
                # print('input label 오류')
                inputs['label'] = 0 
            features.append(inputs)
        
        all_input_ids = torch.tensor([f['input_ids'] for f in features], dtype=torch.long)
        all_attention_mask = torch.tensor([f['attention_mask'] for f in features], dtype=torch.long)
        all_token_type_ids = torch.tensor([f['token_type_ids'] for f in features], dtype=torch.long)
        all_labels = torch.tensor([f['label'] for f in features], dtype=torch.long)

        dataset = TensorDataset(all_input_ids, all_attention_mask, all_token_type_ids, all_labels)
        return dataset
    
    def shuffle_data(self, dataset, data_type):
        if data_type == 'train':
            return RandomSampler(dataset)
        elif data_type == 'eval' or data_type == 'test':
            return SequentialSampler(dataset)
        
    def load_data(self, dataset, sampler):
        return DataLoader(dataset, sampler=sampler, batch_size=self.batch_size)

In [218]:
class BertDataset(Dataset):
    def __init__(self, data_file):
        self.data = data_file
    
    def __len__(self):
        return len(self.data.label)
    
    def reset_index(self):
        self.data.reset_index(inplace=True, drop=True)
    
    def __getitem__(self, idx):
        '''
        return text, label
        '''
        self.reset_index()
        text = self.data.text[idx]
        label = self.data.label[idx]
        return text, label

In [219]:
class BertRegressor(nn.Module):
    def __init__(self, config, model):
        super(BertRegressor, self).__init__()
        self.model = model
        self.linear = nn.Linear(config.hidden_size, 128)
        self.relu = nn.ReLU()
        self.out = nn.Linear(128, 1)
    
    def forward(self, input_ids, attention_mask, token_type_ids):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        logits = outputs.last_hidden_state[:, 0, :]
        # print(f'logits: {len(logits)}, {len(logits[0])}')
        x = self.linear(logits)
        x = self.relu(x)
        score = self.out(x)
        # print(f'score: {score}')
        return score 

In [220]:
class BertRegTester():
    def __init__(self, training_config, model):
        self.training_config = training_config
        self.model = model

    def get_label(self, test_dataloader, test_type):
        '''
        test_type: 0  -> Test dataset 
        test_type: 1  -> Test sentence
        '''
        preds = []
        labels = []

        for batch in test_dataloader:
            self.model.eval()    # self 안 붙이면 이상한 Output (BaseModelOutputWithPoolingAndCrossAttentions) 출력 
            batch = tuple(t.to(self.training_config.device) for t in batch)   # args.device: cuda 
            with torch.no_grad():
                inputs = {
                    "input_ids": batch[0],
                    "attention_mask": batch[1],
                    "token_type_ids": batch[2],
                }
                outputs = self.model(**inputs)
                if test_type == 0:
                    try:
                        preds.extend(outputs.squeeze().detach().cpu().numpy())
                    except:
                        preds.extend(outputs[0].detach().cpu().numpy())
                elif test_type == 1:
                    preds.extend(outputs[0].detach().cpu().numpy())            
            label = batch[3].detach().cpu().numpy()
            labels.extend(label)
        return preds, labels 

In [221]:
def RMSELoss(yhat,y):
    return torch.sqrt(torch.mean((yhat-y)**2))

In [222]:
with open(os.path.join(config_path, 'training_config.json')) as f:
    training_config = AttrDict(json.load(f))

In [223]:
training_config.device = torch.device("cuda") if torch.cuda.is_available() else "cpu"
training_config.config_path = config_path
training_config.model_path = model_path 
training_config.pad = 'max_length'
training_config.num_epochs = 10

In [224]:
tokenizer = BertTokenizer.from_pretrained(os.path.join(base_model, 'bert-base'), model_max_length=128)
config = BertConfig.from_pretrained(os.path.join(base_model, 'bert-base', 'bert_config.json'))

In [225]:
p_model = f'bert_untagged_a{criteria}_s{seed}.pt'

In [226]:
model = BertModel.from_pretrained(os.path.join(base_model, 'bert-base'), config=config)

In [227]:
reg_model = BertRegressor(config, model).to(training_config.device)

In [228]:
config.max_position_embeddings = 128
config.max_position_embeddings

128

In [229]:
reg_model.load_state_dict(torch.load(os.path.join(model_path, p_model)))
reg_model.to(training_config.device)

BertRegressor(
  (model): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0): BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=Tru

In [230]:
bert_processor = BertProcessor(config, training_config, tokenizer)

In [231]:
test_file = BertDataset(test_data)
test_dataset = bert_processor.convert_data(test_file)
test_sampler = bert_processor.shuffle_data(test_dataset, 'test')
test_dataloader = bert_processor.load_data(test_dataset, test_sampler)

In [232]:
bert_tester = BertRegTester(training_config, reg_model)

In [233]:
bws_pred, bws_true = bert_tester.get_label(test_dataloader, 0)

In [234]:
criterion = RMSELoss
criterion(torch.Tensor(bws_pred), torch.Tensor(bws_true))

tensor(1.6330)

In [235]:
bws_pred[-10:], bws_true[-10:]

([7.2613873,
  0.97319466,
  3.5152366,
  7.209349,
  1.8970542,
  5.224246,
  3.8970392,
  1.0413362,
  3.5226119,
  1.6802558],
 [8, 1, 1, 6, 5, 3, 4, 2, 8, 2])

In [236]:
bws_pred = pd.DataFrame(bws_pred, columns=['pred'])
bws_pred.tail(3)

,pred
317,1.041336
318,3.522612
319,1.680256
